### 환경설정

In [ ]:
# open AI API key 등록
import os
OPENAI_API_KEY=""
# 현재 노트북 커널에 환경변수 등록
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY


# LangSmith API key 등록
LANGSMITH_TRACING="true" # 추적 할지 말지
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=""
LANGSMITH_PROJECT="Langchain0422"
# 현재 노트북 커널에 환경변수 등록
os.environ['LANGSMITH_TRACING'] = LANGSMITH_TRACING
os.environ['LANGSMITH_ENDPOINT'] = LANGSMITH_ENDPOINT
os.environ['LANGSMITH_API_KEY'] = LANGSMITH_API_KEY
os.environ['LANGSMITH_PROJECT'] = LANGSMITH_PROJECT

In [2]:
!pip install langchain langchain-openai langchain-community faiss-cpu pypdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180

### 네이버 뉴스기사를 기반으로 답변하는 챗봇 만들기
- 필요시 크롤링 기술이나 크롤링 에이전트 등을 통해서 뉴스 URL을 수집
- 현재는 직접 URL을 선정하고 챗봇에 삽입하는 방식으로 진행

In [5]:
# 웹문서 RAG 도구
import bs4 # 웹문서 파싱도구
from langchain_community.document_loaders import WebBaseLoader # 웹문서 로딩 도구
from langchain_text_splitters import RecursiveCharacterTextSplitter # 재귀적 텍스트 분할 도구
from langchain_openai import OpenAIEmbeddings # 임베딩 도구
from langchain_community.vectorstores import FAISS # 벡터데이터베이스 도구
# Chain 구성을 위한 도구
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿
from langchain.chat_models import init_chat_model # 모델을 생성하는 도구
from langchain_core.output_parsers import StrOutputParser # 문자열 파싱 도구
from langchain_core.runnables import RunnablePassthrough # 데이터 전달용 Runnable

### 1. 뉴스기사 파싱하기

In [8]:
# 파싱할 URL
url = 'https://www.newsis.com/view/NISX20260420_0003598500'

In [9]:
web_loader = WebBaseLoader(
    web_path = [url], # 읽을 웹페이지 경로(여러개 가능)
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            "div", attrs = {"class" : "viewer"}
        )
    )
)

In [10]:
docs = web_loader.load()
docs

[Document(metadata={'source': 'https://www.newsis.com/view/NISX20260420_0003598500'}, page_content="\n\n\n'반반윙박스 20p' 7종 새롭게 선봬\n\n\n[서울=뉴시스] 교촌에프앤비가 교촌의 인기 소스를 더욱 풍성하고 다채롭게 즐길 수 있는 '반반윙박스 20p'를 필두로 윙 제품 라인업 강화에 나섰다.(사진=교촌에프앤비) *재판매 및 DB 금지\r\n[서울=뉴시스]김상윤 기자 = 교촌에프앤비가 교촌의 인기 소스를 더욱 풍성하고 다채롭게 즐길 수 있는 '반반윙박스 20p'를 필두로 윙 제품 라인업 강화에 나섰다고 20일 밝혔다.\n\r\n교촌에프앤비에 따르면 이번 라인업 강화는 기존보다 늘어난 조각 수와 더욱 다양해진 맛 구성으로 소비자 선택의 폭을 획기적으로 넓혔다.\n\r\n교촌은 기존 16조각 구성의 '윙박스 16p'에 4조각을 더해 풍성함을 더한 '윙박스 20p' 7종과 교촌의 인기 소스 두 가지를 반반으로 즐길 수 있는 '반반윙박스 20p' 7종을 새롭게 선보였다. \n\r\n신제품 '반반윙박스 20p'는 ▲간장+레드 ▲간장+마라레드 ▲간장+허니갈릭 ▲레드+마라레드 ▲레드+허니갈릭 ▲마라레드+허니갈릭 ▲후라이드+양념 등 총 7종으로 구성된다.\n\r\n교촌은 늘어난 조각 수만큼 넉넉한 양과 교촌의 시그니처 소스들을 윙 부위로 다채롭게 즐길 수 있는 이번 라인업 확대를 통해 고객들의 세분화된 취향을 적극 공략할 계획이다.\n\n\n◎공감언론 뉴시스 [email\xa0protected]\n\n\n\nCopyright © NEWSIS.COM, 무단 전재 및 재배포 금지\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n")]

### 2. chunk 단위로 쪼개기

In [12]:
# 구분자는 설정하지 않고 기본 값 사용 -> \n\n, \n, " ", ""
news_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 100, # 청크 사이즈
    chunk_overlap = 50, # 청크 오버랩 사이즈
)
news_chunk = news_splitter.split_documents(docs)
len(news_chunk)

11

### 3. 임베딩 후 벡터DB에 저장

In [13]:
# FAISS 활용
faiss_db = FAISS.from_documents(
    news_chunk, # 청크 데이터
    OpenAIEmbeddings() # 임베딩 도구
)

### 4. 검색기 생성

In [14]:
# chunk 사이즈가 작기 때문에 찾아오는 유사청크 갯수를 늘려서 설정
news_retriver = faiss_db.as_retriever(search_kwagrs={"k" : 5})

### 5. Chain 생성 및 호출

In [15]:
template = ChatPromptTemplate.from_messages([
    ('system','''
    너는 뉴스 챗봇이야. 아래 context를 참고해서 question에 대한 답변을 만들어줘.
    모르는 정보는 모른다고 반드시 답변해.

    context : {context}
    '''),
    ('human', 'question : {question}')
])

In [16]:
# chain 구성
news_chain = {"question" : RunnablePassthrough(),
              "context" : news_retriver} | template | init_chat_model("openai:gpt-4o-mini") | StrOutputParser()

In [17]:
print(news_chain.invoke("최근 치킨관련 뉴스 알려줘"))

최근 교촌에프앤비가 '반반윙박스 20p'를 launch하며 윙 제품 라인업을 강화했다고 알려졌습니다. 이 제품은 교촌의 인기 소스 두 가지를 반반으로 즐길 수 있는 7종이 새롭게 선보였습니다. 더 자세한 내용은 [여기](https://www.newsis.com/view/NISX20260420_0003598500)에서 확인하실 수 있습니다.


In [18]:
print(news_chain.invoke("미국 전쟁과 관련된 뉴스 알려줘"))

죄송하지만, 현재 제공된 context에는 미국 전쟁과 관련된 정보가 포함되어 있지 않습니다. 더 구체적인 질문이나 다른 주제에 대해 문의해 주시면 도움이 될 수 있습니다.
